<a href="https://colab.research.google.com/github/Naresh6603/naresh/blob/main/6603iomp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
import gradio as gr
import warnings
warnings.filterwarnings('ignore')

# 2. Load the dataset (Upload your file first!)
print("Please upload your file 'disease_prediction_dataset.csv.xls'")
from google.colab import files
uploaded = files.upload()

# Read the file
df = pd.read_csv('disease_prediction_dataset (3).csv')

print(f"Dataset shape: {df.shape}")
print("Disease distribution:\n", df['disease'].value_counts())

# 3. Preprocessing
# Drop unnecessary column
df = df.drop(['patient_id'], axis=1)

# Encode binary Yes/No columns
binary_cols = ['fever', 'cough', 'fatigue', 'shortness_of_breath', 'chest_pain',
               'headache', 'nausea', 'diarrhea', 'skin_rash', 'joint_pain',
               'frequent_urination', 'blurred_vision', 'family_history', 'smoking', 'alcohol']

for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].map({'Yes': 1, 'No': 0, 'yes':1, 'no':0})

# Encode categorical columns
cat_cols = ['gender', 'blood_pressure', 'cholesterol', 'exercise_frequency',
            'diet_quality', 'city', 'season']

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Target encoding
target_le = LabelEncoder()
df['disease_encoded'] = target_le.fit_transform(df['disease'])

# Features and target
X = df.drop(['disease', 'disease_encoded'], axis=1)
y = df['disease_encoded']

# Scale numerical features
scaler = StandardScaler()
num_cols = ['age', 'bmi', 'blood_sugar', 'hemoglobin', 'symptom_duration_days',
            'num_symptoms', 'stress_level', 'sleep_hours']
X[num_cols] = scaler.fit_transform(X[num_cols])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Train models
print("Training models...")

rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

xgb = XGBClassifier(n_estimators=300, random_state=42, use_label_encoder=False, eval_metric='mlogloss')
xgb.fit(X_train, y_train)

# Evaluate
y_pred_rf = rf.predict(X_test)
y_pred_xgb = xgb.predict(X_test)

print("\nRandom Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("\nClassification Report (XGBoost):\n", classification_report(y_test, y_pred_xgb, target_names=target_le.classes_))

# Use the best model (usually XGBoost performs better here)
model = xgb

# 5. Gradio UI
def predict_disease(age, gender, bmi, blood_pressure, cholesterol, blood_sugar, hemoglobin,
                    fever, cough, fatigue, shortness_of_breath, chest_pain, headache, nausea,
                    diarrhea, skin_rash, joint_pain, frequent_urination, blurred_vision,
                    symptom_duration_days, num_symptoms, family_history, smoking, alcohol,
                    exercise_frequency, diet_quality, stress_level, sleep_hours, city, season):

    # Create input DataFrame
    input_data = pd.DataFrame([{
        'age': age,
        'gender': gender,
        'bmi': bmi,
        'blood_pressure': blood_pressure,
        'cholesterol': cholesterol,
        'blood_sugar': blood_sugar,
        'hemoglobin': hemoglobin,
        'fever': 1 if fever == "Yes" else 0,
        'cough': 1 if cough == "Yes" else 0,
        'fatigue': 1 if fatigue == "Yes" else 0,
        'shortness_of_breath': 1 if shortness_of_breath == "Yes" else 0,
        'chest_pain': 1 if chest_pain == "Yes" else 0,
        'headache': 1 if headache == "Yes" else 0,
        'nausea': 1 if nausea == "Yes" else 0,
        'diarrhea': 1 if diarrhea == "Yes" else 0,
        'skin_rash': 1 if skin_rash == "Yes" else 0,
        'joint_pain': 1 if joint_pain == "Yes" else 0,
        'frequent_urination': 1 if frequent_urination == "Yes" else 0,
        'blurred_vision': 1 if blurred_vision == "Yes" else 0,
        'symptom_duration_days': symptom_duration_days,
        'num_symptoms': num_symptoms,
        'family_history': 1 if family_history == "Yes" else 0,
        'smoking': 1 if smoking == "Yes" else 0,
        'alcohol': 1 if alcohol == "Yes" else 0,
        'exercise_frequency': exercise_frequency,
        'diet_quality': diet_quality,
        'stress_level': stress_level,
        'sleep_hours': sleep_hours,
        'city': city,
        'season': season
    }])

    # Encode categorical features using the same label encoders
    for col in cat_cols:
        if col in input_data.columns:
            input_data[col] = label_encoders[col].transform(input_data[col])

    # Scale numerical features
    input_data[num_cols] = scaler.transform(input_data[num_cols])

    # Predict
    pred_encoded = model.predict(input_data)[0]
    disease_name = target_le.inverse_transform([pred_encoded])[0]

    # Get probability
    proba = model.predict_proba(input_data)[0]
    confidence = float(max(proba)) * 100

    return f"**Predicted Disease: {disease_name}**\n\nConfidence: {confidence:.1f}%"

# Create Gradio Interface
with gr.Blocks(title="Disease Prediction System", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏥 Disease Prediction System")
    gr.Markdown("Enter patient details to predict the most likely disease.")

    with gr.Row():
        with gr.Column():
            age = gr.Slider(1, 100, value=45, label="Age")
            gender = gr.Radio(["Male", "Female"], label="Gender", value="Male")
            bmi = gr.Slider(10, 50, value=25.0, label="BMI")
            blood_pressure = gr.Dropdown(["Normal", "High", "Low", "Borderline"], label="Blood Pressure", value="Normal")
            cholesterol = gr.Dropdown(["Normal", "High", "Borderline"], label="Cholesterol", value="Normal")
            blood_sugar = gr.Slider(50, 400, value=120.0, label="Blood Sugar (mg/dL)")
            hemoglobin = gr.Slider(5, 20, value=13.5, label="Hemoglobin (g/dL)")

        with gr.Column():
            fever = gr.Radio(["Yes", "No"], label="Fever", value="No")
            cough = gr.Radio(["Yes", "No"], label="Cough", value="No")
            fatigue = gr.Radio(["Yes", "No"], label="Fatigue", value="No")
            shortness_of_breath = gr.Radio(["Yes", "No"], label="Shortness of Breath", value="No")
            chest_pain = gr.Radio(["Yes", "No"], label="Chest Pain", value="No")
            headache = gr.Radio(["Yes", "No"], label="Headache", value="No")

    with gr.Row():
        with gr.Column():
            nausea = gr.Radio(["Yes", "No"], label="Nausea", value="No")
            diarrhea = gr.Radio(["Yes", "No"], label="Diarrhea", value="No")
            skin_rash = gr.Radio(["Yes", "No"], label="Skin Rash", value="No")
            joint_pain = gr.Radio(["Yes", "No"], label="Joint Pain", value="No")
            frequent_urination = gr.Radio(["Yes", "No"], label="Frequent Urination", value="No")
            blurred_vision = gr.Radio(["Yes", "No"], label="Blurred Vision", value="No")

        with gr.Column():
            symptom_duration_days = gr.Slider(1, 30, value=7, label="Symptom Duration (days)")
            num_symptoms = gr.Slider(0, 10, value=3, label="Number of Symptoms")
            family_history = gr.Radio(["Yes", "No"], label="Family History of Disease", value="No")
            smoking = gr.Radio(["Yes", "No"], label="Smoking", value="No")
            alcohol = gr.Radio(["Yes", "No"], label="Alcohol Consumption", value="No")
            exercise_frequency = gr.Dropdown(["Never", "Rarely", "Sometimes", "Regular"], label="Exercise Frequency", value="Rarely")
            diet_quality = gr.Dropdown(["Poor", "Average", "Good"], label="Diet Quality", value="Average")
            stress_level = gr.Slider(1, 10, value=5, label="Stress Level (1-10)")
            sleep_hours = gr.Slider(3, 12, value=7.0, label="Sleep Hours")
            city = gr.Dropdown(["Hyderabad", "Mumbai", "Delhi", "Bangalore", "Chennai", "Pune"], label="City", value="Hyderabad")
            season = gr.Dropdown(["Summer", "Monsoon", "Winter", "Post-Monsoon"], label="Season", value="Monsoon")

    predict_btn = gr.Button("🔍 Predict Disease", variant="primary", size="large")

    output = gr.Markdown()

    predict_btn.click(
        fn=predict_disease,
        inputs=[age, gender, bmi, blood_pressure, cholesterol, blood_sugar, hemoglobin,
                fever, cough, fatigue, shortness_of_breath, chest_pain, headache, nausea,
                diarrhea, skin_rash, joint_pain, frequent_urination, blurred_vision,
                symptom_duration_days, num_symptoms, family_history, smoking, alcohol,
                exercise_frequency, diet_quality, stress_level, sleep_hours, city, season],
        outputs=output
    )

# 6. Launch with public link
demo.launch(share=True, debug=True)

Please upload your file 'disease_prediction_dataset.csv.xls'


Saving disease_prediction_dataset.csv to disease_prediction_dataset (3).csv
Dataset shape: (10000, 32)
Disease distribution:
 disease
Diabetes        5191
Hypertension    1802
Dengue           972
Malaria          731
Flu              725
Common Cold      515
Healthy           64
Name: count, dtype: int64
Training models...

Random Forest Accuracy: 0.8485
XGBoost Accuracy: 0.9625

Classification Report (XGBoost):
               precision    recall  f1-score   support

 Common Cold       0.92      0.90      0.91       103
      Dengue       0.95      0.95      0.95       195
    Diabetes       0.97      0.99      0.98      1038
         Flu       0.96      0.87      0.91       145
     Healthy       0.83      0.77      0.80        13
Hypertension       0.96      0.95      0.96       360
     Malaria       0.97      0.95      0.96       146

    accuracy                           0.96      2000
   macro avg       0.94      0.91      0.92      2000
weighted avg       0.96      0.96      0